In [ ]:
# download dataset from the UCI website
!curl -o uci-labelled-sentences.zip https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip

# unzip dataset in Colab
!unzip uci-labelled-sentences.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 84188    0 84188    0     0   600k      0 --:--:-- --:--:-- --:--:--  604k
Archive:  uci-labelled-sentences.zip
replace sentiment labelled sentences/.DS_Store? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace __MACOSX/sentiment labelled sentences/._.DS_Store? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [ ]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Embedding, LSTM
from keras.callbacks import EarlyStopping

In [ ]:
df_list = []

# Yelp
df_yelp = pd.read_csv('sentiment labelled sentences/yelp_labelled.txt', names=['sentence', 'label'], sep='\t')
df_yelp['source'] = 'yelp'
df_list.append(df_yelp)

# Amazon
df_amazon = pd.read_csv('sentiment labelled sentences/amazon_cells_labelled.txt', names=['sentence', 'label'], sep='\t')
df_amazon['source'] = 'amazon'
df_list.append(df_amazon)

# IMDB
df_imdb = pd.read_csv('sentiment labelled sentences/imdb_labelled.txt', names=['sentence', 'label'], sep='\t')
df_imdb['source'] = 'imdb'
df_list.append(df_imdb)

# Concatenate the dataframes
df = pd.concat(df_list)

df.head()

,sentence,label,source
0,Wow... Loved this place.,1,yelp
1,Crust is not good.,0,yelp
2,Not tasty and the texture was just nasty.,0,yelp
3,Stopped by during the late May bank holiday of...,1,yelp
4,The selection on the menu was great and so wer...,1,yelp


In [ ]:
max_features = 5000
tokenizer = Tokenizer(num_words=max_features, split=' ')
tokenizer.fit_on_texts(df['sentence'].values)
X = tokenizer.texts_to_sequences(df['sentence'].values)
X = pad_sequences(X)
y = df['label'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.12)

In [ ]:
from keras.layers import Dropout

def create_model():
  model = Sequential()
  model.add(Embedding(max_features, 128, input_length=X.shape[1]))
  model.add(LSTM(32))
  model.add(Dropout(0.5))
  model.add(Dense(1, activation='sigmoid'))
  model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
  return model

model = create_model()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.fit(X_train, y_train, epochs=6, batch_size=16, validation_data=(X_test, y_test), callbacks = [EarlyStopping(monitor='val_accuracy', min_delta=0.001, patience=2, verbose=1)])

Epoch 1/6
152/152 ━━━━━━━━━━━━━━━━━━━━ 103s 655ms/step - accuracy: 0.6150 - loss: 0.6608 - val_accuracy: 0.7303 - val_loss: 0.5849
Epoch 2/6
152/152 ━━━━━━━━━━━━━━━━━━━━ 141s 654ms/step - accuracy: 0.8598 - loss: 0.3691 - val_accuracy: 0.7909 - val_loss: 0.4513
Epoch 3/6
152/152 ━━━━━━━━━━━━━━━━━━━━ 102s 672ms/step - accuracy: 0.9516 - loss: 0.1513 - val_accuracy: 0.7939 - val_loss: 0.4945
Epoch 4/6
152/152 ━━━━━━━━━━━━━━━━━━━━ 142s 671ms/step - accuracy: 0.9785 - loss: 0.0746 - val_accuracy: 0.8091 - val_loss: 0.5804
Epoch 5/6
152/152 ━━━━━━━━━━━━━━━━━━━━ 101s 667ms/step - accuracy: 0.9880 - loss: 0.0461 - val_accuracy: 0.7939 - val_loss: 0.6478
Epoch 6/6
152/152 ━━━━━━━━━━━━━━━━━━━━ 101s 667ms/step - accuracy: 0.9921 - loss: 0.0305 - val_accuracy: 0.8091 - val_loss: 0.7262
Epoch 6: early stopping


In [ ]:
model.save("uci_sentimentanalysis.h5")

with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.DEFAULT_PROTOCOL)